# Saving and Loading grid_py Graphics

This tutorial discusses strategies for creating persistent representations
of `grid_py` graphics, following the R vignette *Saving and Loading grid Graphics*.

Three approaches are covered:
1. **Python source code** -- the most robust and editable form.
2. **Pickle serialisation of grobs** -- save/load grob objects for later editing.
3. **Device output** -- exporting to image formats (PNG, PDF, SVG).

In [ ]:
import os
import pickle
import tempfile

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import grid_py as gp

## Approach 1 -- Python source code

The most common workflow: write `grid_py` calls in a `.py` file or
notebook cell and re-run them.  The source code *is* the persistent
representation and is fully editable.

In [ ]:
# This cell IS the persistent representation
gp.grid_newpage()
gt = gp.text_grob("Hello from grid_py", name="greeting")
gp.grid_draw(gt)
plt.gcf()

## Approach 2 -- Pickle serialisation of grobs

Grob objects can be serialised with `pickle.dump` and reloaded later.
This lets you share or archive a grob tree without sharing the code
that created it.  The reloaded grob can still be edited using
`edit_grob`, `add_grob`, `remove_grob`, etc.

In [ ]:
# Create a grob
gt = gp.text_grob("hi", name="saved_text")

# Save to a temporary file
tmpdir = tempfile.mkdtemp()
save_path = os.path.join(tmpdir, "mygridplot.pkl")

with open(save_path, "wb") as f:
    pickle.dump(gt, f)

print(f"Grob saved to {save_path}")
print(f"File size: {os.path.getsize(save_path)} bytes")

In [ ]:
# Reload the grob
with open(save_path, "rb") as f:
    gt_loaded = pickle.load(f)

# Draw the reloaded grob
gp.grid_newpage()
gp.grid_draw(gt_loaded)
plt.gcf()

The loaded grob is fully editable:

In [ ]:
# Edit the loaded grob
gt_edited = gp.edit_grob(gt_loaded, gp=gp.Gpar(col="red", fontsize=24))

gp.grid_newpage()
gp.grid_draw(gt_edited)
plt.gcf()

## Saving and loading viewports

Viewport objects can also be pickled.

In [ ]:
vp = gp.Viewport(
    width=0.6, height=0.6,
    xscale=[0, 10], yscale=[-1, 1],
    angle=15,
    name="my_vp",
)

vp_path = os.path.join(tmpdir, "myviewport.pkl")
with open(vp_path, "wb") as f:
    pickle.dump(vp, f)

with open(vp_path, "rb") as f:
    vp_loaded = pickle.load(f)

print(f"Loaded viewport: {vp_loaded}")
print(f"  angle = {vp_loaded.angle}")
print(f"  xscale = {vp_loaded.xscale}")

## Approach 3 -- Device output

Finally, you can export the rendered figure to PNG, PDF, or SVG via
matplotlib's `savefig`.  This is persistent across Python versions but
is not reloadable as an editable grob tree.

In [ ]:
gp.grid_newpage()
gp.grid_grill()
gp.grid_text("Exported figure", 0.5, 0.5, gp=gp.Gpar(fontsize=18))

png_path = os.path.join(tmpdir, "grid_output.png")
plt.savefig(png_path, dpi=100)
print(f"Figure saved to {png_path}")
print(f"File size: {os.path.getsize(png_path)} bytes")

In [ ]:
# Clean up temporary files
import shutil
shutil.rmtree(tmpdir, ignore_errors=True)